
# PHM 2010 Milling Dataset — Full Data Audit Notebook

This notebook guides a thorough, *publishable* data audit for the PHM 2010 Milling dataset.

**How to use**
1. Set `ROOT_DIR` (below) to your dataset folder (e.g., `r"E:\Collaboration Work\With Farooq\phm dataset\PHM Challange 2010 Milling"`).
2. Run cells from top to bottom.
3. Use the generated tables/plots and the final markdown **Auto-Report** for your notes.
4. Outputs (CSVs/PNGs) will be saved under a `./artifacts/` folder next to this notebook.

> This notebook avoids internet and external dependencies. It uses only standard Python packages + matplotlib/pandas/scipy (if available).


In [ ]:

# --- Imports
import os, re, sys, glob, math, json, textwrap, shutil, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# For MAT files if present
try:
    from scipy.io import loadmat
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings('ignore')

# --- Config (EDIT THIS) ---
ROOT_DIR = r"E:\Collaboration Work\With Farooq\phm dataset\PHM Challange 2010 Milling"  # <-- SET THIS to your dataset folder, e.g., r"E:\\Collaboration Work\\With Farooq\\phm dataset\\PHM Challange 2010 Milling"
ARTIFACTS_DIR = Path('./artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# If files are very large, adjust sample size for fast previews
ROW_SAMPLE_LIMIT = 50000  # per file when loading quick stats

# Plot settings (no specific colors/styles to comply with simple matplotlib usage)
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['figure.dpi'] = 120

def ensure_root():
    if not ROOT_DIR or not os.path.isdir(ROOT_DIR):
        raise RuntimeError("Please set ROOT_DIR to your PHM 2010 Milling dataset path.")
    return Path(ROOT_DIR)


## 1. Scan directory structure & file inventory

In [ ]:
root = ensure_root()
print("ROOT_DIR:", root)
all_files = sorted([str(p) for p in Path(root).rglob('*') if p.is_file()])
print(f"Total files found: {len(all_files)}")

ext_counts = Counter([Path(f).suffix.lower() for f in all_files])
print("File types:", ext_counts)

inv_df = pd.DataFrame({
    "path": all_files,
    "ext": [Path(f).suffix.lower() for f in all_files],
    "size_bytes": [os.path.getsize(f) for f in all_files],
})
inv_df["size_mb"] = inv_df["size_bytes"] / (1024*1024)
inv_df.to_csv(ARTIFACTS_DIR / "file_inventory.csv", index=False)
print("Saved inventory to artifacts/file_inventory.csv")
inv_df.head(10)


RuntimeError: Please set ROOT_DIR to your PHM 2010 Milling dataset path.

## 2. Parse metadata from filenames (runs, tools, conditions)

In [ ]:

def parse_meta_from_name(path):
    name = Path(path).name
    name_low = name.lower()
    m = {
        "filename": name,
        "run_id": None,
        "tool_id": None,
        "set": None,  # train/test/validation if present
        "speed": None,
        "feed": None,
        "pass_": None,
    }
    # Heuristics—adjust if your dataset uses different naming
    run_m = re.search(r'run[_\-]?(\d+)', name_low) or re.search(r'train[_\-]?(\d+)', name_low)
    if run_m:
        m["run_id"] = run_m.group(1)
    tool_m = re.search(r'tool[_\-]?(\d+)', name_low)
    if tool_m:
        m["tool_id"] = tool_m.group(1)
    set_m = re.search(r'(train|test|valid|validation)', name_low)
    if set_m:
        m["set"] = set_m.group(1)
    speed_m = re.search(r'(?:speed|rpm)[_\-]?(\d+)', name_low)
    if speed_m:
        m["speed"] = speed_m.group(1)
    feed_m = re.search(r'(?:feed)[_\-]?(\d+)', name_low)
    if feed_m:
        m["feed"] = feed_m.group(1)
    pass_m = re.search(r'(?:pass)[_\-]?(\d+)', name_low)
    if pass_m:
        m["pass_"] = pass_m.group(1)
    return m

meta_records = []
for p in all_files:
    if Path(p).suffix.lower() in {'.csv', '.txt', '.tsv', '.mat'}:
        meta_records.append({**parse_meta_from_name(p), "path": p})

meta_df = pd.DataFrame(meta_records)
meta_df.to_csv(ARTIFACTS_DIR / "filename_metadata.csv", index=False)
print("Saved filename-derived metadata to artifacts/filename_metadata.csv")
meta_df.head(12)


## 3. Inspect tabular signals (columns, sampling, missingness)

In [ ]:

def sniff_delimiter(path, default=','):
    with open(path, 'rb') as f:
        head = f.read(2048).decode('latin-1', errors='ignore')
    # Try tabs/commas/semicolons/spaces
    for d in ['\t', ',', ';', ' ']:
        parts = head.split('\n')[0].split(d)
        if len(parts) > 1:
            return d
    return default

def load_quick(path, nrows=None):
    ext = Path(path).suffix.lower()
    if ext in {'.csv', '.txt', '.tsv'}:
        delim = '\t' if ext == '.tsv' else sniff_delimiter(path)
        try:
            df = pd.read_csv(path, sep=delim, nrows=nrows, engine='python')
        except Exception:
            df = pd.read_csv(path, sep=None, nrows=nrows, engine='python')
        return df
    elif ext == '.mat' and SCIPY_AVAILABLE:
        mat = loadmat(path)
        # Try to promote a common key to DataFrame
        # Heuristic: pick first 2D variable
        for k, v in mat.items():
            if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                df = pd.DataFrame(v)
                df.columns = [f"col_{i}" for i in range(df.shape[1])]
                return df
        return pd.DataFrame()
    else:
        return pd.DataFrame()

stats = []
preview_tables = {}
for p in inv_df["path"].tolist():
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv', '.mat'}:
        continue
    df = load_quick(p, nrows=ROW_SAMPLE_LIMIT)
    if df.empty:
        continue
    cols = list(df.columns)
    nrows, ncols = df.shape
    missing = df.isna().sum().sum()
    stats.append({
        "path": p,
        "nrows_sampled": nrows,
        "ncols": ncols,
        "columns": cols[:50],  # show first 50 names
        "missing_in_sample": int(missing),
    })
    preview_tables[p] = df.head(5)

stats_df = pd.DataFrame(stats)
stats_df.to_csv(ARTIFACTS_DIR / "file_quick_stats.csv", index=False)
print("Saved quick stats to artifacts/file_quick_stats.csv")
stats_df.head(10)


### Preview a few representative files

In [ ]:

sample_paths = list(preview_tables.keys())[:3]
for sp in sample_paths:
    print("\n=== Preview:", sp, "===")
    display(preview_tables[sp])


## 4. Normalize/standardize channel names (heuristic mapping)

In [ ]:

# Map raw column names to canonical names when possible (edit as needed for your dataset)
CANON_MAP = {
    r'fx|force[_\-]?x': 'Fx',
    r'fy|force[_\-]?y': 'Fy',
    r'fz|force[_\-]?z': 'Fz',
    r'ax|acc[_\-]?x|vib[_\-]?x': 'AccX',
    r'ay|acc[_\-]?y|vib[_\-]?y': 'AccY',
    r'az|acc[_\-]?z|vib[_\-]?z': 'AccZ',
    r'ae|acous|emission': 'AE',
    r'rpm|speed': 'RPM',
    r'feed': 'Feed',
    r'temp|temperature': 'Temp',
    r't': 'Time'
}
import re
def canon_name(col):
    cl = str(col).strip().lower()
    for pattern, out in CANON_MAP.items():
        if re.fullmatch(pattern, cl) or re.search(pattern, cl):
            return out
    return str(col)

def quick_canon_columns(df):
    return df.rename(columns={c: canon_name(c) for c in df.columns})

# Try canonicalization on a sample file
if len(preview_tables):
    sp = next(iter(preview_tables))
    df0 = preview_tables[sp]
    display(quick_canon_columns(df0).head(5))


## 5. Global channel census & per-channel stats

In [ ]:

channel_counter = Counter()
per_file_channels = {}
for p in stats_df["path"]:
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv', '.mat'}:
        continue
    df = load_quick(p, nrows=2000)
    if df.empty:
        continue
    df = quick_canon_columns(df)
    cols = list(df.columns)
    per_file_channels[p] = cols
    for c in cols:
        channel_counter[c] += 1

chan_df = pd.DataFrame(channel_counter.items(), columns=["channel", "count_present_in_files"]).sort_values("count_present_in_files", ascending=False)
chan_df.to_csv(ARTIFACTS_DIR / "channel_census.csv", index=False)
print("Saved channel census to artifacts/channel_census.csv")
chan_df.head(20)


## 6. Estimate sampling rate for time-indexed files

In [ ]:

def estimate_fs(df):
    # If 'Time' exists and looks numeric increasing, estimate fs as 1/median dt
    if 'Time' in df.columns:
        t = pd.to_numeric(df['Time'], errors='coerce').dropna().values
        if len(t) > 10:
            dt = np.diff(t)
            dt = dt[dt > 0]
            if len(dt) > 0:
                fs = 1.0 / np.median(dt)
                return fs
    # Otherwise, assume uniform sampling and use index with a guessed rate (unknown)
    return np.nan

fs_records = []
for p in stats_df["path"]:
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv', '.mat'}:
        continue
    df = load_quick(p, nrows=5000)
    if df.empty:
        continue
    df = quick_canon_columns(df)
    fs = estimate_fs(df)
    fs_records.append({"path": p, "fs_est_hz": fs})

fs_df = pd.DataFrame(fs_records)
fs_df.to_csv(ARTIFACTS_DIR / "sampling_rate_estimates.csv", index=False)
print("Saved sampling rate estimates to artifacts/sampling_rate_estimates.csv")
fs_df.head(10)


## 7. Descriptive stats & correlations (per file sample)

In [ ]:

desc_records = []
corr_records = []
N_SAMP = 10000

for p in stats_df["path"][:30]:  # limit to 30 files for speed; increase if needed
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv', '.mat'}:
        continue
    df = load_quick(p, nrows=N_SAMP)
    if df.empty:
        continue
    df = quick_canon_columns(df).select_dtypes(include=[np.number])
    if df.shape[1] < 2:
        continue
    d = df.describe().T
    d["path"] = p
    d["column"] = d.index
    desc_records.append(d.reset_index(drop=True))

    corr = df.corr().reset_index().melt(id_vars="index", var_name="col2", value_name="corr")
    corr = corr.rename(columns={"index": "col1"})
    corr["path"] = p
    corr_records.append(corr)

desc_df = pd.concat(desc_records, ignore_index=True) if desc_records else pd.DataFrame()
corr_df = pd.concat(corr_records, ignore_index=True) if corr_records else pd.DataFrame()

desc_df.to_csv(ARTIFACTS_DIR / "descriptive_stats_sample.csv", index=False)
corr_df.to_csv(ARTIFACTS_DIR / "correlations_sample.csv", index=False)

print("Saved descriptive stats to artifacts/descriptive_stats_sample.csv")
print("Saved correlations to artifacts/correlations_sample.csv")

display(desc_df.head(10))


## 8. Quick plots: time series, PSD, STFT (sample file)

In [ ]:

def plot_timeseries(df, cols, title, out_png):
    plt.figure()
    for c in cols:
        if c in df.columns:
            plt.plot(df[c].values)
    plt.title(title)
    plt.xlabel("Sample Index")
    plt.ylabel("Signal Value")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.show()

def plot_psd_welch(df, cols, fs, title, out_png):
    from scipy.signal import welch
    plt.figure()
    for c in cols:
        if c in df.columns:
            x = pd.to_numeric(df[c], errors='coerce').dropna().values
            if len(x) > 1024:
                f, Pxx = welch(x, fs=fs if not math.isnan(fs) else 1.0, nperseg=1024)
                plt.semilogy(f, Pxx)
    plt.title(title + f" (fs~{fs:.2f} Hz)" if not math.isnan(fs) else title + " (fs unknown)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.show()

def plot_stft(df, col, fs, title, out_png):
    from scipy.signal import stft
    x = pd.to_numeric(df[col], errors='coerce').dropna().values
    if len(x) < 2048:
        return
    f, t, Zxx = stft(x, fs=fs if not math.isnan(fs) else 1.0, nperseg=256, noverlap=128)
    plt.figure()
    plt.pcolormesh(t, f, np.abs(Zxx), shading='gouraud')
    plt.title(title)
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (s)')
    plt.tight_layout()
    plt.savefig(out_png)
    plt.show()

# Choose a candidate file to visualize
sample_file = None
for p in stats_df["path"]:
    if Path(p).suffix.lower() in {'.csv', '.txt', '.tsv'}:
        sample_file = p
        break

if sample_file is not None:
    dfv = load_quick(sample_file, nrows=40000)
    dfv = quick_canon_columns(dfv)
    fs_est = estimate_fs(dfv)

    # Pick up to 3 numeric channels
    num_cols = [c for c in dfv.columns if pd.api.types.is_numeric_dtype(dfv[c])]
    plot_cols = num_cols[:3]

    plot_timeseries(dfv, plot_cols, "Sample Time Series (first 3 numeric channels)", ARTIFACTS_DIR / "plot_timeseries.png")
    try:
        plot_psd_welch(dfv, plot_cols, fs_est, "PSD via Welch", ARTIFACTS_DIR / "plot_psd_welch.png")
        if len(plot_cols) > 0:
            plot_stft(dfv, plot_cols[0], fs_est, f"STFT - {plot_cols[0]}", ARTIFACTS_DIR / "plot_stft.png")
    except Exception as e:
        print("Skipping PSD/STFT due to:", e)
else:
    print("No sample .csv/.txt file found for visualization.")


## 9. Windowing utility & small parquet export (for modeling)

In [ ]:

def make_windows(df, cols, win_len=1024, stride=512):
    X = []
    for start in range(0, len(df)-win_len+1, stride):
        seg = df[cols].iloc[start:start+win_len].to_numpy()
        X.append(seg)
    return np.stack(X) if len(X) else np.empty((0, win_len, len(cols)))

# Build a mini modeling set from first few files
model_rows = []
MAX_FILES = 5
count = 0
for p in stats_df["path"]:
    if count >= MAX_FILES: 
        break
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv'}:
        continue
    df = load_quick(p, nrows=60000)
    if df.empty:
        continue
    df = quick_canon_columns(df)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) < 3:
        continue
    X = make_windows(df, num_cols[:3], win_len=1024, stride=512)
    if X.size == 0:
        continue
    for i in range(min(5, X.shape[0])):  # save a few windows per file
        model_rows.append({
            "path": p,
            "cols": num_cols[:3],
            "window_idx": i,
            "array": X[i].tolist()
        })
    count += 1

mini_df = pd.DataFrame(model_rows)
mini_path = ARTIFACTS_DIR / "mini_windows.parquet"
try:
    mini_df.to_parquet(mini_path, index=False)
    print("Saved mini modeling windows to", mini_path)
except Exception as e:
    csv_path = ARTIFACTS_DIR / "mini_windows.csv"
    mini_df.drop(columns=["array"]).to_csv(csv_path, index=False)
    print("Parquet unavailable; saved CSV (without arrays) to", csv_path)


## 10. Build a channel graph (k-NN by correlation)

In [ ]:

def channel_graph_from_corr(df, k=3):
    df_num = df.select_dtypes(include=[np.number])
    if df_num.shape[1] < 2:
        return [], []
    cols = list(df_num.columns)
    C = df_num.corr().values
    edges = set()
    for i in range(len(cols)):
        # Get top-k neighbors by absolute corr (excluding self)
        idx = np.argsort(-np.abs(C[i]))  # descending
        added = 0
        for j in idx:
            if i == j: 
                continue
            edges.add(tuple(sorted((i, j))))
            added += 1
            if added >= k:
                break
    nodes = list(range(len(cols)))
    return nodes, sorted(edges)

graph_records = []
for p in stats_df["path"][:10]:  # limit for speed
    if Path(p).suffix.lower() not in {'.csv', '.txt', '.tsv'}:
        continue
    df = load_quick(p, nrows=20000)
    if df.empty:
        continue
    df = quick_canon_columns(df)
    nodes, edges = channel_graph_from_corr(df, k=3)
    graph_records.append({
        "path": p,
        "nodes": nodes,
        "edges": edges,
        "channels": [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    })

pd.DataFrame(graph_records).to_json(ARTIFACTS_DIR / "channel_graphs.json", orient="records", indent=2)
print("Saved channel graphs to artifacts/channel_graphs.json (nodes are indices into 'channels')")


## 11. Auto-Report (Markdown)

In [ ]:

summary = []
summary.append("# PHM 2010 Milling — Data Audit Report\n")
summary.append("## Inventory\n")
summary.append(f"- Total files scanned: **{len(all_files)}**\n")
summary.append(f"- File types: **{dict(ext_counts)}**\n")
summary.append("\n## Common Channels (Top 20)\n")
if 'chan_df' in globals() and not chan_df.empty:
    summary.append(chan_df.head(20).to_markdown(index=False))
else:
    summary.append("_No channel census available._")
summary.append("\n\n## Sampling Rate Estimates (first 10)\n")
if 'fs_df' in globals() and not fs_df.empty:
    summary.append(fs_df.head(10).to_markdown(index=False))
else:
    summary.append("_No sampling rate estimates available._")
summary.append("\n\n## Descriptive Stats Snapshot\n")
if 'desc_df' in globals() and not desc_df.empty:
    summary.append(desc_df.head(20).to_markdown(index=False))
else:
    summary.append("_No descriptive stats available._")
summary.append("\n\n## Artifacts\n")
summary.append("- `artifacts/file_inventory.csv`\n- `artifacts/filename_metadata.csv`\n- `artifacts/file_quick_stats.csv`\n- `artifacts/channel_census.csv`\n- `artifacts/sampling_rate_estimates.csv`\n- `artifacts/descriptive_stats_sample.csv`\n- `artifacts/correlations_sample.csv`\n- `artifacts/plot_timeseries.png`\n- `artifacts/plot_psd_welch.png` (if SciPy available)\n- `artifacts/plot_stft.png` (if SciPy available)\n- `artifacts/mini_windows.parquet` (or CSV fallback)\n- `artifacts/channel_graphs.json`\n")

report_path = ARTIFACTS_DIR / "PHM2010_Audit_Report.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(summary))

print("Wrote report:", report_path)



---

### Next steps (after audit)
- Confirm which columns map to **forces (Fx,Fy,Fz)**, **vibration/AE**, and **operating conditions (RPM/Feed)**.
- Choose consistent sampling rates or resample as needed.
- Lock in window length/stride for modeling.
- Use `channel_graphs.json` to initialize a GNN edge list (k ~ 2–4 works well).

> When you're happy with the audit, we can scaffold the modeling code (baselines + the monotone GNN) on top of the exported `mini_windows` and your finalized preprocessing.
